# E and B modes against survey property maps (DP2, PSZ2 G309.43-72.86, tract 2877)

Alex Broughton · October 2026. Rubin stack and Butler access required.

## Introduction

Compares the Schirmer-filtered aperture E- and B-mode S/N maps of tract 2877, which contains PSZ2 G309.43-72.86, with
i-band survey property maps sampled on the same HEALPix cells (NSIDE = 4096). A trend of the E mode with a survey
property points to a systematic. The B mode should show no trend with any property.

- Mass map. Read from output/2877_shapeHSM_massmap.fits, written by dp2_shearmap_PSZ2_G309.43-72.86.ipynb. Only cells
  with aperture completeness of at least 0.6 are used, the same threshold as for peaks.
- Survey property maps. i-band HealSparse consolidated maps, degraded on read to NSIDE = 4096 and sampled at the mass
  map cell centers. Loading follows _tutorials/survey-property-maps.ipynb.
- Binned means. Cells are correlated over the 18′ aperture, so the error bars on the binned means underestimate the
  true uncertainty.

## 1.0 Setup

### 1.1 Imports and configuration

All tunable quantities live in the CONFIG dictionary.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from scipy.stats import binned_statistic

from lsst.daf.butler import Butler

%matplotlib inline
plt.rcParams.update({"font.size": 12})


# Repo layout: _data/ holds external inputs, output/ holds generated tables/catalogs, plots/ holds figures.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = ROOT / "_data"
OUTPUT_DIR = ROOT / "output"
PLOTS_DIR = ROOT / "plots"
for _d in (OUTPUT_DIR, PLOTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)


def save_plot(fig, name):
    fig.savefig(PLOTS_DIR / f"{name}.pdf", bbox_inches="tight")


def mark_cluster(ax, color="k"):
    """Mark CLUSTER on an RA/Dec axis with a star."""
    ax.scatter(CLUSTER["ra"], CLUSTER["dec"], marker="*", s=250, facecolors="none",
               edgecolors=color, linewidths=1.5, zorder=5)

In [ ]:
CONFIG = {
    "REPO": "/sdf/data/rubin/repo/dp2_prep",  # Butler repository root
    "COLLECTION": "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",  # Input collection
    "SKYMAP": "lsst_cells_v2",  # Skymap name for Butler
    "INSTRUMENT": "LSSTCam",  # Instrument dimension
    "TRACT": 2877,  # Contains PSZ2 G309.43-72.86
    "MASSMAP": OUTPUT_DIR / "2877_shapeHSM_massmap.fits",  # E/B map from dp2_shearmap_PSZ2_G309.43-72.86.ipynb
    "NSIDE": 4096,  # Mass map NSIDE; survey property maps are degraded to this on read
    "COMPLETENESS_MIN": 0.6,  # Minimum aperture completeness, as for peaks
    "BAND": "i",  # Band of the survey property maps (shapes are i-band HSM)
    # Survey property maps: axis label -> dataset type.
    "SP_MAPS": {
        "PSF size (pixel)": "deepCoadd_psf_size_consolidated_map_weighted_mean",
        "PSF $e_1$": "deepCoadd_psf_e1_consolidated_map_weighted_mean",
        "PSF $e_2$": "deepCoadd_psf_e2_consolidated_map_weighted_mean",
        "PSF 5σ mag limit (AB)": "deepCoadd_psf_maglim_consolidated_map_weighted_mean",
        "Exposure time (s)": "deepCoadd_exposure_time_consolidated_map_sum",
        "Sky background (nJy)": "deepCoadd_sky_background_consolidated_map_weighted_mean",
    },
    "N_BINS": 10,  # Quantile bins of each survey property for the binned means
}

# Target cluster.
CLUSTER = {"name": "PSZ2 G309.43-72.86", "ra": 10.2082, "dec": -44.1307}

In [ ]:
butler = Butler(
    CONFIG["REPO"],
    collections=[CONFIG["COLLECTION"]],
    skymap=CONFIG["SKYMAP"],
    instrument=CONFIG["INSTRUMENT"],
)

## 2.0 Load data

### 2.1 E and B mode maps

One row per evaluated HEALPix cell, with the cell center, aperture values, S/N, and completeness.

In [ ]:
mmap = Table.read(CONFIG["MASSMAP"])
assert mmap.meta["NSIDE"] == CONFIG["NSIDE"]

good = (mmap["COMPLETENESS"] >= CONFIG["COMPLETENESS_MIN"]) & np.isfinite(mmap["E_SN"])
mmap = mmap[good]

ra = np.asarray(mmap["RA"])
dec = np.asarray(mmap["DEC"])
e_sn = np.asarray(mmap["E_SN"])
b_sn = np.asarray(mmap["B_SN"])
print(f"{len(mmap)} of {len(good)} cells with completeness >= {CONFIG['COMPLETENESS_MIN']}")

### 2.2 Survey property maps

Each map is degraded on read to the mass map NSIDE and sampled at the cell centers. Cells outside a map's footprint are
set to NaN.

In [ ]:
sp = {}
for label, dataset_type in CONFIG["SP_MAPS"].items():
    hsp = butler.get(dataset_type, band=CONFIG["BAND"], parameters={"degrade_nside": CONFIG["NSIDE"]})
    valid = hsp.get_values_pos(ra, dec, valid_mask=True)
    sp[label] = np.where(valid, hsp.get_values_pos(ra, dec), np.nan)
    print(f"{dataset_type}: nside_sparse={hsp.nside_sparse}, {valid.mean():.1%} of cells covered")

## 3.0 Survey property maps on the tract

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5), constrained_layout=True)
for ax, (label, x) in zip(axes.flat, sp.items()):
    lo, hi = np.nanpercentile(x, [2, 98])
    sc = ax.scatter(ra, dec, c=x, s=8, vmin=lo, vmax=hi, linewidths=0)
    fig.colorbar(sc, ax=ax, label=label)
    mark_cluster(ax)
    ax.set_xlabel("RA (deg)")
    ax.set_ylabel("Dec (deg)")
    ax.invert_xaxis()  # RA increases to the left
fig.suptitle(f"Tract {CONFIG['TRACT']}, {CONFIG['BAND']}-band survey properties, NSIDE={CONFIG['NSIDE']}")
save_plot(fig, f"{CONFIG['TRACT']}_survey_property_maps")

## 4.0 E and B modes against survey properties

Each panel shows the cell S/N against one survey property, with the mean S/N in quantile bins of the property and the
Pearson correlation coefficient r over all cells.

In [ ]:
def plot_against_sp(y, ylabel):
    fig, axes = plt.subplots(2, 3, figsize=(15, 8.5), constrained_layout=True)
    for ax, (label, x) in zip(axes.flat, sp.items()):
        ok = np.isfinite(x) & np.isfinite(y)
        edges = np.unique(np.quantile(x[ok], np.linspace(0, 1, CONFIG["N_BINS"] + 1)))
        mean, _, _ = binned_statistic(x[ok], y[ok], "mean", edges)
        std, _, _ = binned_statistic(x[ok], y[ok], "std", edges)
        n, _, _ = binned_statistic(x[ok], y[ok], "count", edges)

        ax.hexbin(x[ok], y[ok], gridsize=40, mincnt=1, cmap="Greys", bins="log")
        ax.errorbar(0.5 * (edges[1:] + edges[:-1]), mean, std / np.sqrt(n), fmt="o", color="C3")
        ax.axhline(0.0, color="k", linewidth=0.8, linestyle="--")
        ax.set_xlabel(label)
        ax.set_ylabel(ylabel)
        ax.set_title(f"r = {np.corrcoef(x[ok], y[ok])[0, 1]:.3f}")
    fig.suptitle(f"Tract {CONFIG['TRACT']}, {ylabel} against {CONFIG['BAND']}-band survey properties")
    return fig

### 4.1 E mode

In [ ]:
fig = plot_against_sp(e_sn, "E-mode S/N")
save_plot(fig, f"{CONFIG['TRACT']}_E-mode_vs_survey_properties")

### 4.2 B mode

A trend here indicates a systematic, since lensing produces no B mode.

In [ ]:
fig = plot_against_sp(b_sn, "B-mode S/N")
save_plot(fig, f"{CONFIG['TRACT']}_B-mode_vs_survey_properties")